# 👀 Lab 1: Ask My Notes — Build a RAG Chatbot Over THIS Week's Lectures

The LLM doesn't know what happened in this classroom this week. Let's prove that — and then FIX it, by giving the model eyes: **Retrieval-Augmented Generation**.

## 📚 The plan
1. 🙈 Prove the model doesn't know our course
2. 🔎 Build a retriever over the week's notes (yesterday's semantic search, new job!)
3. 🧩 RAG = retrieve + paste into prompt + answer
4. 📌 Sources, and the honest "I don't know"

🔑 Needs your OpenRouter API or OpenAI API key (from your instructor). Never hardcode keys!

In [1]:
!pip install openai sentence-transformers

### 🛠️ Setup

In [2]:
!pip install openai sentence-transformers -q
!pip install rank_bm25

from openai import OpenAI
from getpass import getpass
from sentence_transformers import SentenceTransformer, util

api_key = getpass("🔑 Paste your OpenRouter API key: ")
client = OpenAI(api_key=api_key)


🔑 Paste your OpenRouter API key: ··········


In [3]:
MODEL = "gpt-5.4-nano"
def ask(prompt, system=None, temperature=0.3, timeout=15):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        timeout=timeout
    )
    return response.choices[0].message.content

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("👀 Ready to give the model eyes.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

👀 Ready to give the model eyes.


### 🙈 Section 1: Prove the Problem

Ask the model about OUR course. It wasn't in the training data — watch what happens.

In [4]:
print(ask("In the Majal Initiative AI track, what lab did students do on Day 2, and what dataset was used?"))

I don’t have enough context to answer this reliably. “Majal Initiative AI track” could refer to different cohorts/materials, and “Day 2” plus the specific lab/dataset isn’t something I can uniquely identify without the exact program page or schedule.

If you share a link, screenshot, or the Day 2 agenda text for the AI track, I can tell you exactly:
1) what lab students did on Day 2, and  
2) which dataset was used.


It either admits ignorance or — worse — **invents a confident answer** (hallucination, exactly as Day 4 theory predicted: plausible ≠ true). The model needs OUR notes. Let's give them to it.

### 📚 Section 2: The Knowledge Base — This Week's Notes

In real RAG systems, documents get split into **chunks**. Ours are short summaries of the week — one chunk per topic. *(In production these would come from PDFs, wikis, databases... same idea, bigger scale.)*

In [5]:
notes = [
    "Day 1 covered AI foundations and supervised learning: features, labels, and the train/test split.",
    "Day 1 lab: students trained a linear regression to predict calories burned from workout duration and heart rate.",
    "Day 1 lab: a logistic regression classified mushrooms as edible or poisonous; recall mattered most because eating a poisonous mushroom (a false negative) is the worst mistake.",
    "Day 1 explained overfitting with the student analogy: memorizing practice questions instead of learning, detected by a gap between training and test accuracy.",
    "Day 2 covered unsupervised learning: finding structure in data with no labels.",
    "Day 2 lab: K-Means clustered 200 mall customers by annual income and spending score into five segments; the elbow method chose K.",
    "Day 2 lab: students compressed their own photos with K-Means by clustering pixel colors into K palette colors.",
    "Day 2 compared K-Means and DBSCAN: DBSCAN finds clusters of any shape and labels lonely points as noise, which suits fraud detection.",
    "Day 3 covered computer vision and deep learning: CNNs use small sliding filters, pooling, and stacked layers that learn edges, then textures, then objects.",
    "Day 3 lab: students built a CNN in PyTorch for cats vs dogs on CIFAR-10 and visualized the filters the network invented.",
    "Day 3 lab: the digit recognizer showed the augmentation trap: flipping and rotating digits made 6 and 9 collapse into each other on the confusion matrix.",
    "Day 3 lab: transfer learning with a frozen ResNet18 and a new 2-class head crushed the from-scratch CNN on the same cats and dogs data.",
    "Day 4 covered NLP: tokenization, bag of words and its blindness to order and meaning, then word embeddings placing similar words close together.",
    "Day 4 explained contextual embeddings with the bank example: the word bank gets different vectors near money versus near river.",
    "Day 4 explained attention with the trophy and suitcase sentence, and LLMs as next-token predictors trained on internet-scale text.",
    "Day 4 lab: students called an LLM through the OpenRouter API, changed its personality with system prompts, and tested temperature 0 versus 1.3.",
    "The instructor of the AI track is Fadel, and each day had a theory session followed by hands-on labs with Golden Questions.",
]

note_vecs = embedder.encode(notes)
print(f"📚 Knowledge base ready: {len(notes)} chunks, each embedded into {note_vecs.shape[1]} numbers.")

📚 Knowledge base ready: 17 chunks, each embedded into 384 numbers.


### 🔎 Section 3: The Retriever — Yesterday's Search Engine, New Job

This is *literally* the semantic search you built in Day 4 Lab 2: embed the question, cosine-compare against all chunks, return the top matches.

In [6]:
def retrieve(question, top_k=5):
    q_vec = embedder.encode([question])
    scores = util.cos_sim(q_vec, note_vecs)[0].clone()
    # keyword boost: if a note shares a distinctive word with the query, nudge its score up
    q_words = set(question.lower().replace("?","").split())
    for i, note in enumerate(notes):
        overlap = q_words & set(note.lower().split())
        # boost distinctive words (day numbers, dataset names), not stopwords
        for w in overlap:
            if w.isdigit() or w in {"day", "cifar-10", "mnist", "mushroom", "mall"}:
                scores[i] += 0.15
    best = scores.argsort(descending=True)[:top_k]
    return [(float(scores[i]), notes[i]) for i in best]

for score, chunk in retrieve("what did we do with mall customers?"):
    print(f"({score:.2f}) {chunk}")

(0.62) Day 2 lab: K-Means clustered 200 mall customers by annual income and spending score into five segments; the elbow method chose K.
(0.20) Day 2 compared K-Means and DBSCAN: DBSCAN finds clusters of any shape and labels lonely points as noise, which suits fraud detection.
(0.14) Day 2 covered unsupervised learning: finding structure in data with no labels.
(0.14) Day 4 explained contextual embeddings with the bank example: the word bank gets different vectors near money versus near river.
(0.12) Day 4 covered NLP: tokenization, bag of words and its blindness to order and meaning, then word embeddings placing similar words close together.


### 🎯 Mini Exercise 3.1
Retrieve for: *"which mistake is worse when classifying dangerous food?"* — no shared keywords with any note. Does it still find the mushroom chunk? Why? (You answered this yesterday 😉)

In [13]:
# TO-DO: run the retrieval and inspect the results
for score, chunk in retrieve("which mistakes is worse when classifying dangerousss foood?"):
    print(f"({score:.2f}) {chunk}")

(0.54) Day 1 lab: a logistic regression classified mushrooms as edible or poisonous; recall mattered most because eating a poisonous mushroom (a false negative) is the worst mistake.
(0.26) Day 1 explained overfitting with the student analogy: memorizing practice questions instead of learning, detected by a gap between training and test accuracy.
(0.22) Day 2 compared K-Means and DBSCAN: DBSCAN finds clusters of any shape and labels lonely points as noise, which suits fraud detection.
(0.21) Day 4 explained attention with the trophy and suitcase sentence, and LLMs as next-token predictors trained on internet-scale text.
(0.18) Day 2 covered unsupervised learning: finding structure in data with no labels.


### 🧩 Section 4: Assemble RAG — Retrieve, Augment, Generate

Now the whole pipeline in one function: retrieve the best chunks, paste them into the prompt as **context**, and instruct the model to answer ONLY from that context — including saying "I don't know" when the context doesn't cover it.

In [8]:
RAG_SYSTEM = """You are a helpful course assistant for the Field Initiative AI track.
Answer the question using ONLY the provided context.
If the context does not contain the answer, say "I don't know based on the course notes." — do NOT guess.
Keep answers to 2-4 sentences."""


def rag_answer(question, top_k=3, show_sources=True):
    # call the retrieval function to get the top_k most
    results = retrieve(question, top_k)

    # stitch the retrieved chunks into a single bulleted string,
    context = "\n".join(f"- {chunk}" for _, chunk in results)

    # build the actual prompt sent to the model: retrieved context first,
    # then the question, so the model has grounding material before it answers
    prompt = f"Context:\n{context}\n\nQuestion: {question}"

    # send the prompt with the RAG system prompt attached, temperature=0
    # keeps answers deterministic/factual rather than creative
    answer = ask(prompt, system=RAG_SYSTEM, temperature=0)

    print(f"❓ {question}\n")
    print(f"🤖 {answer}")

    if show_sources:
        print("\n📌 Sources used:")
        for score, chunk in results:
            # truncate each chunk to 90 chars so the source list stays scannable
            print(f"   ({score:.2f}) {chunk[:90]}...")

rag_answer("What lab did we do on Day 2 and what dataset was used?")

❓ What lab did we do on Day 2 and what dataset was used?

🤖 On Day 2, we did labs on unsupervised learning using K-Means. One lab had students compress their own photos by clustering pixel colors into K palette colors, and another lab clustered 200 mall customers by annual income and spending score into five segments (with K chosen using the elbow method). The dataset used was the mall customers dataset with annual income and spending score.

📌 Sources used:
   (0.63) Day 2 lab: students compressed their own photos with K-Means by clustering pixel colors in...
   (0.60) Day 2 lab: K-Means clustered 200 mall customers by annual income and spending score into f...
   (0.60) Day 2 covered unsupervised learning: finding structure in data with no labels....


In [9]:
# The SAME question that failed in Section 1 — now grounded. Compare!
rag_answer("Why did 6 and 9 get confused in the digit lab?")

❓ Why did 6 and 9 get confused in the digit lab?

🤖 6 and 9 got confused because the digit recognizer hit the augmentation trap: flipping and rotating digits made 6 and 9 collapse into each other on the confusion matrix.

📌 Sources used:
   (0.95) Day 3 lab: the digit recognizer showed the augmentation trap: flipping and rotating digits...
   (0.25) Day 3 lab: students built a CNN in PyTorch for cats vs dogs on CIFAR-10 and visualized the...
   (0.23) Day 4 explained contextual embeddings with the bank example: the word bank gets different ...


In [10]:
# The honesty test: ask something NOT in the notes
rag_answer("What is the price of the workshop's lunch menu?")

❓ What is the price of the workshop's lunch menu?

🤖 I don't know based on the course notes.

📌 Sources used:
   (0.27) Day 2 lab: K-Means clustered 200 mall customers by annual income and spending score into f...
   (0.18) Day 4 lab: students called an LLM through the OpenRouter API, changed its personality with...
   (0.12) Day 2 lab: students compressed their own photos with K-Means by clustering pixel colors in...


**Look at that last one.** A raw LLM might invent a lunch menu. Your RAG bot said it doesn't know — because you constrained it to the retrieved context. That single behavior is why companies deploy RAG: **grounded answers, sources shown, honest gaps.**

### 🎯 Mini Exercise 4.1 — Stress-Test Your Bot
1. Ask 3 questions in your own words (different phrasing than the notes!) and check the sources it used.
2. Try `top_k=1` on a question that needs info from TWO days. What breaks? What does that teach about choosing top_k?
3. Add 3 new note chunks about anything (your hobbies, a fake Day 6...) and ask about them.

In [17]:
# TO-DO: your stress tests


# --- Part 1: three questions in our own words (phrasing NOT in the notes) ---
# print("="*60, "\nPART 1: Paraphrased questions\n", "="*60)
# rag_answer("")

# print("\n" + "-"*60 + "\n")
# rag_answer("")

# print("\n" + "-"*60 + "\n")
# rag_answer("")

print("="*60, "\nPART 1: Paraphrased questions\n", "="*60)
rag_answer("What method helped divide shoppers into groups based on how much they earn and spend?")

print("\n" + "-"*60 + "\n")
rag_answer("Why was missing a poisonous food item considered the most serious error?")

print("\n" + "-"*60 + "\n")
rag_answer("What is my name ?")

PART 1: Paraphrased questions
❓ What method helped divide shoppers into groups based on how much they earn and spend?

🤖 The lab used **K-Means clustering** to divide shoppers into groups based on their **annual income** and **spending score**.

📌 Sources used:
   (0.45) Day 2 lab: K-Means clustered 200 mall customers by annual income and spending score into f...
   (0.22) Day 2 lab: students compressed their own photos with K-Means by clustering pixel colors in...
   (0.14) Day 2 compared K-Means and DBSCAN: DBSCAN finds clusters of any shape and labels lonely po...

------------------------------------------------------------

❓ Why was missing a poisonous food item considered the most serious error?

🤖 Missing a poisonous food item was considered the most serious error because it could lead to eating a poisonous mushroom, which is the worst mistake (a false negative).

📌 Sources used:
   (0.54) Day 1 lab: a logistic regression classified mushrooms as edible or poisonous; recall matt

In [15]:
# --- Part 2: top_k=1 on a question that needs TWO chunks ---
# question = ""

# print(">>> With top_k=1 (only ONE chunk retrieved):")
# rag_answer(question, top_k=)

# print("\n>>> With top_k=3 (room for both facts):")
# rag_answer(question, top_k=)

question = "Compare the dangerous-food classifier from Day 1 with the digit augmentation problem from Day 3."

print(">>> With top_k=1 (only ONE chunk retrieved):")
rag_answer(question, top_k=1)

print("\n>>> With top_k=3 (room for both facts):")
rag_answer(question, top_k=3)

>>> With top_k=1 (only ONE chunk retrieved):
❓ Compare the dangerous-food classifier from Day 1 with the digit augmentation problem from Day 3.

🤖 The Day 1 dangerous-food classifier used logistic regression and emphasized **recall** because the worst error was a **false negative** (eating a poisonous mushroom). The Day 3 digit augmentation problem focuses on **augmenting digits** to improve the training data, rather than on choosing a metric like recall to control a specific type of mistake.

📌 Sources used:
   (0.76) Day 1 lab: a logistic regression classified mushrooms as edible or poisonous; recall matte...

>>> With top_k=3 (room for both facts):
❓ Compare the dangerous-food classifier from Day 1 with the digit augmentation problem from Day 3.

🤖 The Day 1 dangerous-food (mushroom) classifier emphasized that some errors are more costly: a false negative (predicting edible when it’s poisonous) is the worst mistake, so recall mattered most. In contrast, the Day 3 digit augmentation 

In [16]:
# --- Part 3: add 3 new chunks, THEN rebuild the index (critical!) ---

# notes.extend([
#     "",
#     "",
#     "",
# ])


# # ⚠️ MUST re-embed after changing notes, or the index is stale and retrieval breaks!
# note_vecs = embedder.encode(notes)
# print(f"📚 Knowledge base rebuilt: {len(notes)} chunks\n")

# rag_answer("")


# print("\n" + "-"*60 + "\n")


# rag_answer("")


notes.extend([
    "Day 6 covered responsible AI: fairness, privacy, transparency, and reducing bias.",
    "Day 6 lab: students built a simple RAG chatbot that answered questions using university policy documents.",
    "Day 6 Golden Question asked why a RAG system must show sources and admit when information is missing.",
])

# ⚠️ MUST re-embed after changing notes, or the index is stale and retrieval breaks!
note_vecs = embedder.encode(notes)
print(f"📚 Knowledge base rebuilt: {len(notes)} chunks\n")

rag_answer("What ideas were discussed in the responsible AI lesson?")

print("\n" + "-"*60 + "\n")

rag_answer("What did students build on Day 6 and why should it show its evidence?")

📚 Knowledge base rebuilt: 20 chunks

❓ What ideas were discussed in the responsible AI lesson?

🤖 In the responsible AI lesson (Day 6), the course discussed fairness, privacy, transparency, and reducing bias.

📌 Sources used:
   (0.58) Day 6 covered responsible AI: fairness, privacy, transparency, and reducing bias....
   (0.49) The instructor of the AI track is Fadel, and each day had a theory session followed by han...
   (0.43) Day 1 covered AI foundations and supervised learning: features, labels, and the train/test...

------------------------------------------------------------

❓ What did students build on Day 6 and why should it show its evidence?

🤖 On Day 6, students built a simple RAG chatbot that answered questions using university policy documents. It should show its evidence (sources) because a RAG system must provide transparency and admit when information is missing, so users can verify the answers and understand limitations.

📌 Sources used:
   (0.70) Day 6 Golden Ques

## The GOLDEN Question 🏆

A hospital wants an assistant that answers questions about its internal treatment protocols — documents that get updated **every month** and where a wrong answer is dangerous.

**Argue why RAG beats fine-tuning here, using three specific properties you just built: updating, grounding, and sources. Then: what is the ONE remaining weakness — what happens if the retriever brings back the WRONG chunk?** 🤔

*(Hold that last thought — the agent in Lab 2 will learn to recover from bad tool results...)*